# Code Verification by the Method of Manufactured Solutions

This notebook presents the Method of Manufactured Solutions (MMS) used for code verification and provides a walkthrough of the code that implements it.

Verification and Validation answer two distinct questions [[1]](#ref1)[[7]](#ref7): 

- **Verification** Does the code solve its equations correctly, at the order of accuracy the discretisation claims?
- **Validation**  Are these the right equations for the physical problem?

In short, verification is *solving the equations right*, and validation is *solving the right equations* [[4]](#ref4).

Verifying a code means comparing its solution against an exact one. The problem with classical exact solutions is that they exist
only for very simple geometries, loads and material laws, and even then they leave terms of the
equations untested [[1]](#ref1)[[3]](#ref3). A code can pass them and still be wrong. 
MMS removes this limitation: it builds the problem around a solution we construct, so an exact
solution is always available.

## The MMS in a nutshell

Suppose we wish to solve the mathematical model $\mathcal{L}(u) = 0$, where $\mathcal{L}$ is the
differential operator and $u$ the unknown. The main idea is to choose a manufactured solution
$\hat{u}$ first, then apply the governing equations to it: what remains, $s = \mathcal{L}(\hat{u})$,
is the source term. By construction, $\hat{u}$ is the exact solution of the modified model
[[5]](#ref5)

$$\mathcal{L}(u) = s$$

The boundary conditions are derived from $\hat{u}$ as well. Three properties make this practical
[[5]](#ref5):

- $s$ depends on the coordinates only, not on $u$. It enters the code as a known load, that is, as
  input.
- Since $s$ is a known function, it needs no discrete version of $\mathcal{L}$.
- The left-hand side $\mathcal{L}(u)$ is unchanged, so the discretisation under test is not
  modified; only its input is.

In short, the expectation is that solving the modified problem gives back $\hat{u}$.

In practice, the code does not solve $\mathcal{L}(u) = s$ itself but a discrete version of it: a
finite element code solves its weak form on a mesh. Its solution $u_h$ is then only an
approximation of $\hat{u}$, and the difference $e = u_h - \hat{u}$ is the error. Because $\hat{u}$
is known exactly, this error can be computed rather than estimated.

### Measuring the error

The error has several sources [[2]](#ref2):

$$e = \varepsilon_h + \varepsilon_\text{ro} + \varepsilon_\text{it} + \varepsilon_\text{bug}$$

- $\varepsilon_h$, the *discretisation error*, which comes from the mesh and vanishes as it is
  refined;
- $\varepsilon_\text{ro}$, the *round-off error*, from finite-precision arithmetic;
- $\varepsilon_\text{it}$, the *iterative error*, from stopping the linear or Newton solver at a
  finite tolerance;
- $\varepsilon_\text{bug}$, the error caused by *coding mistakes*.

Under a correct implementation $\varepsilon_\text{bug} = 0$, and only $\varepsilon_h$ should remain,
provided $\varepsilon_\text{ro}$ and $\varepsilon_\text{it}$ are kept far below it.

The error is a field over $\Omega$, so its size is measured with norms integrated over the domain:

$$\|e\|_{L^2} = \left(\int_\Omega |e|^2\,d\Omega\right)^{1/2}, \qquad
  |e|_{H^1} = \left(\int_\Omega \|\nabla e\|^2\,d\Omega\right)^{1/2}$$

The $L^2$ norm measures the error itself, the $H^1$ semi-norm its gradient.

Order verification assumes that the discretisation error can be expanded in powers of the mesh size
$h$ [[5]](#ref5):

$$\|\varepsilon_h\| = C\,h^p + O(h^{p+1})$$

$p$ is the formal order of accuracy of the discretisation. $C$ does not depend on $h$, but on the
higher derivatives of the manufactured solution [[5]](#ref5). As $h$ decreases, the higher-order
terms vanish faster than the first one, so on fine enough meshes the leading term dominates and
$\|\varepsilon_h\| \approx C\,h^p$. For elements whose shape functions are polynomials of degree
$q$, $p = q + 1$ in the $L^2$ norm and $p = q$ in the $H^1$ semi-norm [[8]](#ref8).

$\varepsilon_\text{ro}$ and $\varepsilon_\text{it}$ do not decrease with $h$. They set a noise
floor, and an error below it must not be used to measure the order: the ratio of two such errors
reflects round-off and solver tolerance, not the discretisation [[2]](#ref2). A coding mistake
shows up as an error that does not vanish, or that vanishes at the wrong rate.

The code is judged by the rate at which the error decreases, not by its size. A single error value
on a single mesh does not tell whether the code is correct: how small is small enough depends on the
solution, the units and the mesh, all of which enter $C$, while $p$ depends on the discretisation
alone. The mesh is therefore refined by a factor $r = h_\text{coarse}/h_\text{fine} > 1$, and the
ratio of the two errors gives the observed order of accuracy [[5]](#ref5):

$$\hat{p} = \frac{\ln\!\left(\|e_{rh}\| / \|e_{h}\|\right)}{\ln r}$$

The test passes when $\hat{p}$ matches $p$. Checking the order is a much stronger criterion than
checking that the error is small, or even that it decreases [[3]](#ref3).

Everything under test is implemented in SOFA, in C++: the force fields
(`LinearSmallStrainFEMForceField`), the
solvers, and the components that turn a body force into nodal loads (`VectorSourceTerm`,
`FEMSourceTermIntegrator`). The suite does not modify any of them; they are the discrete
$\mathcal{L}$ whose left-hand side stays unchanged.

The Python scripts do everything around them:

- derive the source term $s$ and the boundary conditions from $\hat{u}$ and the material law;
- apply them to a SOFA scene as loads and constraints;
- solve the scene on a sequence of refined meshes, measure the error on each and read its order.

### Further reading

- Roache [[1]](#ref1): a short review of MMS for code verification by one of its originators.
- Roy [[2]](#ref2): a review of code and solution verification, with a six-step procedure
  and guidance on choosing manufactured solutions.
- Salari and Knupp [[3]](#ref3): the most complete treatment, including the blind test in which
  coding mistakes were planted in a code to measure what MMS detects.
- Oberkampf and Roy [[5]](#ref5), Chapters 5 and 6: the textbook reference on code verification
  and MMS.
- Aycock *et al.* [[6]](#ref6): MMS applied to elastostatic solid mechanics in a finite element
  solver, the closest precedent for this suite.

<a id="references"></a>
## References

<a id="ref1"></a>
**[1]** P. J. Roache, "Code verification by the method of manufactured solutions,"
*Journal of Fluids Engineering*, 124(1):4&ndash;10, 2002.
[doi:10.1115/1.1436090](https://doi.org/10.1115/1.1436090)

<a id="ref2"></a>
**[2]** C. J. Roy, "Review of code and solution verification procedures for computational
simulation," *Journal of Computational Physics*, 205(1):131&ndash;156, 2005.
[doi:10.1016/j.jcp.2004.10.036](https://doi.org/10.1016/j.jcp.2004.10.036)

<a id="ref3"></a>
**[3]** K. Salari and P. Knupp, *Code Verification by the Method of Manufactured Solutions*,
SAND2000-1444, Sandia National Laboratories, Albuquerque, NM, June 2000.

<a id="ref4"></a>
**[4]** S. Étienne, A. Garon and D. Pelletier, "Some manufactured solutions for verification of
fluid-structure interaction codes," *Computers and Structures*, 106&ndash;107:56&ndash;67, 2012.
[doi:10.1016/j.compstruc.2012.04.006](https://doi.org/10.1016/j.compstruc.2012.04.006)

<a id="ref5"></a>
**[5]** W. L. Oberkampf and C. J. Roy, *Verification and Validation in Scientific Computing*,
Cambridge University Press, 2010. ISBN 978-0-521-11360-1. &mdash; *Chapter 5 (code verification and
order verification) and Chapter 6, §6.3 (MMS).*

<a id="ref6"></a>
**[6]** K. I. Aycock, N. Rebelo and B. A. Craven, "Method of manufactured solutions code
verification of elastostatic solid mechanics problems in a commercial finite element solver,"
*Computers and Structures*, 229:106175, 2020.
[doi:10.1016/j.compstruc.2019.106175](https://doi.org/10.1016/j.compstruc.2019.106175)

<a id="ref7"></a>
**[7]** W. L. Oberkampf and T. G. Trucano, "Verification and validation in computational fluid
dynamics," *Progress in Aerospace Sciences*, 38(3):209&ndash;272, 2002.
[doi:10.1016/S0376-0421(02)00005-2](https://doi.org/10.1016/S0376-0421%2802%2900005-2)

<a id="ref8"></a>
**[8]** P. G. Ciarlet, *The Finite Element Method for Elliptic Problems*, North-Holland, Amsterdam,
1978. Reprinted by SIAM, Classics in Applied Mathematics 40, 2002.